## Feature Selection

### Feature Mapping & Selection Matrix

The table below details every column in the engineered dataset, specifying whether it is included in feature matrix $X$, assigned as target variable $y$, or excluded as a non-predictive identifier.

| Column Name | Status in Pipeline | Reason / Justification |
| :--- | :--- | :--- |
| `image_name` | **Excluded** | File identifier string; not a predictive feature. |
| `filepath` | **Excluded** | Absolute file system path string; not a predictive feature. |
| `age` | **Target ($y$)** | Continuous target variable for analysis and future prediction. |
| `gender_0`, `gender_1` | **Included in $X$** | One-hot encoded categorical gender indicator variables. |
| `race_0` to `race_4` | **Included in $X$** | One-hot encoded categorical ethnicity indicator variables. |
| `eye_distance` | **Included in $X$** | Raw extracted inter-ocular Euclidean distance (pixels). |
| `mouth_width` | **Included in $X$** | Raw extracted mouth width Euclidean distance (pixels). |
| `face_width` | **Included in $X$** | Raw extracted face bounding box width (pixels). |
| `face_height` | **Included in $X$** | Raw extracted face bounding box height (pixels). |
| `face_aspect_ratio` | **Included in $X$** | Engineered scale-invariant ratio (`face_height / face_width`). |
| `eye_to_face_ratio` | **Included in $X$** | Engineered scale-invariant ratio (`eye_distance / face_width`). |
| `mouth_to_face_ratio` | **Included in $X$** | Engineered scale-invariant ratio (`mouth_width / face_width`). |

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Import pandas for dataset loading, feature selection, and correlation analysis
import pandas as pd

# Load engineered dataset from previous phase
engineered_csv_path = "data/processed/utkface_engineered.csv"
df = pd.read_csv(engineered_csv_path)

print(f"Successfully loaded '{engineered_csv_path}' ({len(df)} rows, {len(df.columns)} columns).")

Successfully loaded 'data/processed/utkface_engineered.csv' (2998 rows, 17 columns).


In [2]:
# Define explicit list of feature columns for X
feature_columns = [
    'gender_0', 'gender_1', 'race_0', 'race_1', 'race_2', 'race_3', 'race_4',
    'eye_distance', 'mouth_width', 'face_width', 'face_height',
    'face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio'
]

# Separate feature matrix X and target vector y
X = df[feature_columns]
y = df['age']

print(f"X shape (rows, features): {X.shape}")
print(f"y shape (target instances): {y.shape}")
print("\nSelected Feature Columns in X:")
print(X.columns.tolist())

X shape (rows, features): (2998, 14)
y shape (target instances): (2998,)

Selected Feature Columns in X:
['gender_0', 'gender_1', 'race_0', 'race_1', 'race_2', 'race_3', 'race_4', 'eye_distance', 'mouth_width', 'face_width', 'face_height', 'face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio']


In [3]:
# Define continuous numeric features (excluding binary dummy columns)
numeric_features = [
    'eye_distance', 'mouth_width', 'face_width', 'face_height',
    'face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio'
]

# Compute linear correlation coefficients against target variable age
correlations = df[numeric_features + ['age']].corr()['age'].drop('age')

# Sort correlations by absolute magnitude in descending order
sorted_correlations = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

print("Correlation of numeric features with age (NOT causation, just linear association strength):")
print(sorted_correlations)

Correlation of numeric features with age (NOT causation, just linear association strength):
face_aspect_ratio      0.254254
mouth_to_face_ratio    0.240711
eye_to_face_ratio      0.212070
face_width            -0.186453
face_height            0.168051
mouth_width            0.110632
eye_distance          -0.011489
Name: age, dtype: float64


### Linear Association Observations

*Placeholder for user observations on linear correlation strengths:*
- `face_aspect_ratio` demonstrates the strongest positive linear association with age ($r = 0.2542$).
- `mouth_to_face_ratio` ($r = 0.2407$) and `eye_to_face_ratio` ($r = 0.2121$) also show moderate positive linear associations with age.
- `eye_distance` shows the weakest overall linear correlation with age ($r = -0.0115$).

*Note:* Full distribution analysis, non-linear relationships, and visual EDA will be conducted in Phase 12.

In [4]:
# Export X and y separately to CSV files
x_output_path = "data/processed/X_features.csv"
y_output_path = "data/processed/y_target.csv"

X.to_csv(x_output_path, index=False)
y.to_csv(y_output_path, index=False)

print(f"Feature matrix X saved successfully to '{x_output_path}' ({X.shape[0]} rows, {X.shape[1]} columns).")
print(f"Target vector y saved successfully to '{y_output_path}' ({len(y)} rows).")

Feature matrix X saved successfully to 'data/processed/X_features.csv' (2998 rows, 14 columns).
Target vector y saved successfully to 'data/processed/y_target.csv' (2998 rows).
